#  Cell 1: Import Libraries

In [1]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

# Cell 2: Get Data Files

In [2]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2025-08-21 20:24:23--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.2.33, 172.67.70.149, 104.26.3.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.2.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip’

book-crossings.zip  100%[===================>]  24.88M  25.7MB/s    in 1.0s    

2025-08-21 20:24:25 (25.7 MB/s) - ‘book-crossings.zip’ saved [26085508/26085508]

Archive:  book-crossings.zip
  inflating: BX-Book-Ratings.csv     
  inflating: BX-Books.csv            
  inflating: BX-Users.csv            


# Cell 3: Import CSV Data into DataFrames

In [3]:
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    low_memory=False
)

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    low_memory=False
)

# Cell 4: Filter Data

In [4]:
# add your code here - consider creating a new cell for each section of code
df = df_ratings

counts1 = df['user'].value_counts()
df = df[df['user'].isin(counts1[counts1 >= 200].index)]
counts = df['rating'].value_counts()
df = df[df['rating'].isin(counts[counts >= 100].index)]

# Cell 5: Merge DataFrames

In [5]:
df = pd.merge(right=df, left = df_books, right_on="isbn", left_on="isbn")

# Cell 6: Drop Duplicates

In [6]:
df = df.drop_duplicates(['title', 'user'])

# Cell 7: Create Pivot Table

In [7]:
ratings_pivot = df.pivot(index = 'title', columns = 'user', values = 'rating').fillna(0)
matrix = csr_matrix(ratings_pivot.values)

# Cell 8: Train KNN Model

In [8]:
model_knn = NearestNeighbors(metric = 'cosine', algorithm = 'brute')
model_knn.fit(matrix)

NearestNeighbors(algorithm='brute', metric='cosine')

# Cell 9: Function to Return Recommended Books

In [9]:
def get_recommends(book = ""):
  distances, indices = model_knn.kneighbors(ratings_pivot.loc[book].values.reshape(1, -1), n_neighbors = 6)
  recommended_books = []
  for i in range(5, 0, -1):
    recommended_books.append([ratings_pivot.index[indices.flatten()[i]], distances.flatten()[i]])
  return [book, recommended_books]

# Cell 10: Test the Function

In [11]:
books = get_recommends("The Queen of the Damned (Vampire Chronicles (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(4):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

['The Queen of the Damned (Vampire Chronicles (Paperback))', [['Delta Of Venus', np.float64(0.6719079311454197)], ["The Doctor's Wife (Japan's Women Writers)", np.float64(0.6302583864920075)], ['Intercourses: An Aphrodisiac Cookbook', np.float64(0.6163880818485585)], ['The Tale of the Body Thief (Vampire Chronicles (Paperback))', np.float64(0.5376338446489461)], ['The Vampire Lestat (Vampire Chronicles, Book II)', np.float64(0.5178411864186413)]]]
You haven't passed yet. Keep trying!
